# E-Commerce Sales Performance Analysis using Data Analytics and AI

**Student:** Paras

This project demonstrates data cleaning, exploratory analysis, visualization, statistical analysis, machine learning, model evaluation and AI/ML-based business insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
pd.set_option("display.max_columns", None)

In [ ]:
df = pd.read_csv("../data/ecommerce_sales_dataset.csv")
print("Raw shape:", df.shape)
df.head()

## 1. Data Understanding

In [ ]:
df.info()
df.describe(include="all").T

## 2. Data Cleaning

In [ ]:
print("Duplicates before:", df.duplicated().sum())
print(df.isna().sum())
df = df.drop_duplicates().copy()
df["Customer_Rating"] = df["Customer_Rating"].fillna(df["Customer_Rating"].median())
df["Discount"] = df["Discount"].fillna(df["Discount"].median())
df["Region"] = df["Region"].fillna(df["Region"].mode()[0])
print("Clean shape:", df.shape)
print("Missing after cleaning:", df.isna().sum().sum())

## 3. Exploratory Data Analysis

In [ ]:
df.groupby("Category")["Sales_INR"].sum().sort_values(ascending=False).plot(kind="bar",figsize=(9,5),title="Sales by Product Category")
plt.ylabel("Sales (INR)"); plt.xticks(rotation=30,ha="right"); plt.show()

In [ ]:
df.groupby("Region")["Sales_INR"].sum().sort_values(ascending=False).plot(kind="bar",figsize=(8,5),title="Sales by Region")
plt.ylabel("Sales (INR)"); plt.show()

In [ ]:
df.groupby("Channel")["Sales_INR"].sum().sort_values(ascending=False).plot(kind="bar",figsize=(8,5),title="Sales by Channel")
plt.ylabel("Sales (INR)"); plt.show()

In [ ]:
monthly=df.assign(Order_Date=pd.to_datetime(df["Order_Date"])).set_index("Order_Date").resample("ME")["Sales_INR"].sum()
monthly.plot(figsize=(10,5),title="Monthly Sales Trend"); plt.ylabel("Sales (INR)"); plt.show()

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(df["Discount"],df["Sales_INR"],alpha=.35)
plt.title("Discount vs Sales"); plt.xlabel("Discount"); plt.ylabel("Sales (INR)"); plt.show()

## 4. Statistical Analysis

In [ ]:
numeric_cols=["Quantity","Unit_Price_INR","Discount","Shipping_Cost_INR","Customer_Rating","Returned","Sales_INR"]
df[numeric_cols].describe().T

In [ ]:
corr=df[numeric_cols].corr()
sns.heatmap(corr,annot=True,fmt=".2f",cmap="Blues")
plt.title("Correlation Heatmap"); plt.show()

## 5. Machine Learning — Sales Prediction

In [ ]:
features=["Region","Category","Channel","Customer_Segment","Payment_Method","Quantity","Unit_Price_INR","Discount","Shipping_Cost_INR","Customer_Rating","Returned"]
target="Sales_INR"
cats=["Region","Category","Channel","Customer_Segment","Payment_Method"]
nums=["Quantity","Unit_Price_INR","Discount","Shipping_Cost_INR","Customer_Rating","Returned"]
pre=ColumnTransformer([
    ("cat",Pipeline([("imp",SimpleImputer(strategy="most_frequent")),("ohe",OneHotEncoder(handle_unknown="ignore"))]),cats),
    ("num",Pipeline([("imp",SimpleImputer(strategy="median")),("scale",StandardScaler())]),nums)
])
X_train,X_test,y_train,y_test=train_test_split(df[features],df[target],test_size=.2,random_state=42)

In [ ]:
lr=Pipeline([("prep",pre),("model",LinearRegression())])
rf=Pipeline([("prep",pre),("model",RandomForestRegressor(n_estimators=180,random_state=42,max_depth=14,min_samples_leaf=2,n_jobs=-1))])
lr.fit(X_train,y_train); rf.fit(X_train,y_train)
p1=lr.predict(X_test); p2=rf.predict(X_test)
def evaluate(name,y,p):
    print(name)
    print("MAE:",round(mean_absolute_error(y,p),2))
    print("RMSE:",round(np.sqrt(mean_squared_error(y,p)),2))
    print("R²:",round(r2_score(y,p),4))
evaluate("Linear Regression",y_test,p1)
evaluate("Random Forest",y_test,p2)

## 6. Actual vs Predicted Sales

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(y_test,p2,alpha=.45)
lo=min(y_test.min(),p2.min()); hi=max(y_test.max(),p2.max())
plt.plot([lo,hi],[lo,hi],"--")
plt.title("Actual vs Predicted Sales — Random Forest")
plt.xlabel("Actual Sales (INR)"); plt.ylabel("Predicted Sales (INR)"); plt.show()

## 7. Feature Importance

In [ ]:
prep=rf.named_steps["prep"]; model=rf.named_steps["model"]
names=list(prep.named_transformers_["cat"].named_steps["ohe"].get_feature_names_out(cats))+nums
importance=pd.Series(model.feature_importances_,index=names).sort_values(ascending=False).head(12)
importance

In [ ]:
importance.sort_values().plot(kind="barh",figsize=(9,5),title="Top Feature Importances")
plt.xlabel("Importance"); plt.show()

## 8. AI/ML-Based Insights

In [ ]:
print("Top category:",df.groupby("Category")["Sales_INR"].sum().idxmax())
print("Top region:",df.groupby("Region")["Sales_INR"].sum().idxmax())
print("Top channel:",df.groupby("Channel")["Sales_INR"].sum().idxmax())
print("Average order sales:",round(df["Sales_INR"].mean(),2))
print("Return rate:",round(df["Returned"].mean()*100,2),"%")

## Conclusion

This project demonstrates an end-to-end Data Analytics and AI workflow for e-commerce sales. The dataset is synthetic and the results are intended for educational and internship demonstration purposes.